In [1]:
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

In [16]:
class DocRequestGenerator:
  """
  Manages the state of the Google Docs API batchUpdate requests.
  Tracks the 'current_index' to ensure formatting is applied to the correct characters.
  """

  def __init__(self):
    self.requests = []
    self.current_index = 1

  def add_text(self, text):
    """
    Creates an insertText request and returns the length of the string
    to help update the global index.
    """
    self.requests.append({
      'insertText': {
        'location': {'index': self.current_index},
        'text': text
      }
    })
    return len(text)

  def apply_heading(self, length, level):
    """
    Applies a Headings to a specific
    range of text based on the length of the header.
    """
    self.requests.append({
      'updateParagraphStyle': {
        'range': {'startIndex': self.current_index, 'endIndex': self.current_index + length},
        'paragraphStyle': {'namedStyleType': f'HEADING_{level}'},
        'fields': 'namedStyleType'
      }
    })

  def apply_list_style(self, length, is_checkbox=False):
    """
    Creates a list structure. Uses 'BULLET_CHECKBOX' for action items
    and standard 'BULLET_DISC' for regular notes.
    """
    self.requests.append({
      'createBullet': {
        'range': {'startIndex': self.current_index, 'endIndex': self.current_index + length},
        'bulletPreset': 'BULLET_CHECKBOX' if is_checkbox else 'BULLET_DISC_CIRCLE_SQUARE'
      }
    })

  def apply_mention_styling(self, text):
    """
    Scans a string for '@name' patterns using regex and applies
    bolding and custom blue coloring to those specific indices.
    """
    for mention in re.finditer(r'@\w+', text):
      start = self.current_index + mention.start()
      end = self.current_index + mention.end()
      self.requests.append({
          'updateTextStyle': {
              'range': {'startIndex': start, 'endIndex': end},
              'textStyle': {
                  'bold': True,
                  'foregroundColor': {'color': {'rgbColor': {'blue': 0.8, 'red': 0.1, 'green': 0.1}}}
              },
              'fields': 'bold,foregroundColor'
          }
      })

In [22]:
class MarkdownParser:
  """
  Interprets Markdown syntax and delegates formatting tasks to
  the DocRequestGenerator.
  """

  def __init__(self, generator):
    self.gen = generator

  def parse_line(self, line):
    """
    The main routing function that identifies the type of Markdown line
    (Heading, List, Checkbox, etc.) and calls the appropriate handler.
    """

    if line.strip() == '---':
      self._handle_separator()
      return

    if not line.strip():
      return

    # Identification Logic
    if line.startswith('# '):
      self._handle_heading(line, 1)
    elif line.startswith('## '):
      self._handle_heading(line, 2)
    elif line.startswith('### '):
      self._handle_heading(line, 3)
    elif "- [ ]" in line:
      self._handle_list_item(line, is_checkbox=True)
    elif line.strip().startswith(('*', '-')):
      self._handle_list_item(line, is_checkbox=False)
    elif line.startswith('---'):
      self._handle_plain_text("\n")
    else:
      self._handle_plain_text(line + "\n")

  def _handle_separator(self):
    """Injected when '---' is encountered: adds a newline to create visual space."""
    length = self.gen.add_text("\n\n")
    self.gen.current_index += length

  def _handle_heading(self, line, level):
    """Removes markdown hashes and applies Heading styling."""
    clean_text = line.replace('#' * level, '').strip() + "\n"
    length = self.gen.add_text(clean_text)
    self.gen.apply_heading(length, level)
    self.gen.current_index += length

  def _handle_list_item(self, line, is_checkbox):
    """Cleans bullet/checkbox syntax and applies list formatting and mentions."""
    clean_text = re.sub(r'^[\s]*([-*]|-\s\[\s\])\s', '', line).strip() + "\n"
    length = self.gen.add_text(clean_text)
    self.gen.apply_list_style(length, is_checkbox)
    self.gen.apply_mention_styling(clean_text)
    self.gen.current_index += length

  def _handle_plain_text(self, text):
    """Handles text that doesn't fit a specific MD category"""
    length = self.gen.add_text(text)
    self.gen.apply_mention_styling(text)
    self.gen.current_index += length

In [28]:
def create_and_format_doc(doc_title, md_content):
    """
    High-level orchestrator:
    1. Initializes the Docs API service
    2. Creates a blank document
    3. Runs the parser to build a list of batchUpdate requests
    4. Executes all formatting in a single API call
    """
    try:
      print("Authenticating...")
      auth.authenticate_user()

      service = build('docs', 'v1')

      # Create the Google Doc
      doc = service.documents().create(body={'title': doc_title}).execute()
      doc_id = doc.get('documentId')

      # Initialize the generator
      generator = DocRequestGenerator()
      parser = MarkdownParser(generator)

      # Process line by line
      for line in md_content.split('\n'):
          parser.parse_line(line)

      # Apply all changes at once using batchUpdate
      if generator.requests:
        service.documents().batchUpdate(
            documentId=doc_id,
            body={'requests': generator.requests}
        ).execute()

      print(f"Success! Document created.")
      print(f"Link: https://docs.google.com/document/d/{doc_id}/edit")
    except HttpError as err:
      print(f"API Error: {err}")

In [25]:
# --- Execution ---
meeting_notes = """# Product Team Sync - May 15, 2023

## Attendees
- Sarah Chen (Product Lead)
- Mike Johnson (Engineering)
- Anna Smith (Design)
- David Park (QA)

## Agenda

### 1. Sprint Review
* Completed Features
  * User authentication flow
  * Dashboard redesign
  * Performance optimization
    * Reduced load time by 40%
    * Implemented caching solution
* Pending Items
  * Mobile responsive fixes
  * Beta testing feedback integration

### 2. Current Challenges
* Resource constraints in QA team
* Third-party API integration delays
* User feedback on new UI
  * Navigation confusion
  * Color contrast issues

### 3. Next Sprint Planning
* Priority Features
  * Payment gateway integration
  * User profile enhancement
  * Analytics dashboard
* Technical Debt
  * Code refactoring
  * Documentation updates

## Action Items
- [ ] @sarah: Finalize Q3 roadmap by Friday
- [ ] @mike: Schedule technical review for payment integration
- [ ] @anna: Share updated design system documentation
- [ ] @david: Prepare QA resource allocation proposal

## Next Steps
* Schedule individual team reviews
* Update sprint board
* Share meeting summary with stakeholders

## Notes
* Next sync scheduled for May 22, 2023
* Platform demo for stakeholders on May 25
* Remember to update JIRA tickets

---
Meeting recorded by: Sarah Chen
Duration: 45 minutes
"""

In [29]:
# --- Execute ---
create_and_format_doc("Product Team Sync - May 15", meeting_notes)

Authenticating...


Success! Document created.
Link: https://docs.google.com/document/d/1Bh1vapAScszI-X1V98SvrI0juYRwiI1mzeAgD-KhYFo/edit
